[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/05_machine_learning_fundamentals/08_distribution_shift.ipynb)

# Bài 08 — Distribution Shift: khi dữ liệu thật khác dữ liệu train

**🎯 Mục tiêu**
- Mọi model ML ngầm giả định: dữ liệu khi triển khai **cùng phân phối** với dữ liệu train (giả định i.i.d.).
- Phân biệt 3 dạng shift: **covariate shift**, **label (prior) shift**, **concept drift**.
- Đo tác động của từng dạng lên model bằng mô phỏng.
- Phát hiện shift bằng **adversarial validation** và kiểm định thống kê; các hướng xử lý.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/05_machine_learning_fundamentals

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, mean_absolute_error, roc_auc_score
from sklearn.model_selection import cross_val_predict

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Ba dạng shift

Viết phân phối dữ liệu là $P(X, Y) = P(Y \mid X)\,P(X)$:

| Dạng | Cái gì thay đổi | Ví dụ |
|---|---|---|
| **Covariate shift** | $P(X)$ thay đổi, $P(Y \mid X)$ giữ nguyên | model chẩn đoán train trên bệnh nhân trẻ, triển khai cho người già; ảnh train chụp ban ngày, triển khai ban đêm |
| **Label (prior) shift** | $P(Y)$ thay đổi, $P(X \mid Y)$ giữ nguyên | tỉ lệ gian lận tăng gấp 3 mùa mua sắm; dịch bệnh làm tỉ lệ dương tính tăng |
| **Concept drift** | $P(Y \mid X)$ thay đổi — **quy luật** thay đổi | hành vi mua sắm sau COVID; kẻ gian đổi chiêu thức; luật thuế mới |

## 2. Covariate shift — model gặp vùng dữ liệu chưa thấy

Quan hệ thật $y = \sin(x)$ không đổi, nhưng train chỉ có $x \in [0, 3]$, còn khi triển khai $x \in [3, 6]$:

In [ ]:
x_train = rng.uniform(0, 3, 200)
y_train = np.sin(x_train) + rng.normal(0, 0.1, 200)
x_deploy = rng.uniform(3, 6, 200)
y_deploy = np.sin(x_deploy) + rng.normal(0, 0.1, 200)

lin = LinearRegression().fit(x_train[:, None], y_train)
grid = np.linspace(0, 6, 300)
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(x_train, y_train, s=10, label="train")
ax.scatter(x_deploy, y_deploy, s=10, label="khi triển khai")
ax.plot(grid, np.sin(grid), "k--", label="quy luật thật (không đổi)")
ax.plot(grid, lin.predict(grid[:, None]), "r", lw=2, label="model")
ax.legend(fontsize=8)
ax.set_title(f"MAE train = {mean_absolute_error(y_train, lin.predict(x_train[:, None])):.2f} | "
             f"MAE khi triển khai = {mean_absolute_error(y_deploy, lin.predict(x_deploy[:, None])):.2f}")
plt.show()

Model chỉ "đúng" trong vùng đã thấy; ngoài vùng đó nó **ngoại suy** sai. Covariate shift đặc biệt nguy hiểm khi model bị ép trả lời cho dữ liệu nằm ngoài vùng train.

## 3. Label shift — xác suất dự đoán bị lệch

Model phát hiện gian lận train khi tỉ lệ gian lận là 2%; mùa mua sắm tỉ lệ thật lên 10%. Hành vi của từng loại giao dịch không đổi ($P(X \mid Y)$ giữ nguyên), nhưng **xác suất dự đoán** bị đánh giá thấp có hệ thống:

In [ ]:
def sample(n, fraud_rate):
    y = (rng.random(n) < fraud_rate).astype(int)
    X = np.where(y[:, None] == 1, rng.normal([2.0, 1.5], 1.0, (n, 2)), rng.normal([0, 0], 1.0, (n, 2)))
    return X, y


X_tr, y_tr = sample(20000, 0.02)
X_new, y_new = sample(20000, 0.10)
clf = LogisticRegression().fit(X_tr, y_tr)
p_new = clf.predict_proba(X_new)[:, 1]
print(f"Tỉ lệ gian lận thật lúc triển khai: {y_new.mean():.3f} | trung bình xác suất model dự đoán: {p_new.mean():.3f}")
print(f"ROC-AUC vẫn tốt: {roc_auc_score(y_new, p_new):.3f} — thứ hạng không đổi, nhưng xác suất bị lệch")

# Hiệu chỉnh prior: odds_mới = odds_cũ × (π_mới/(1−π_mới)) / (π_cũ/(1−π_cũ))
odds = p_new / (1 - p_new) * (0.10 / 0.90) / (0.02 / 0.98)
p_adj = odds / (1 + odds)
print(f"Sau khi hiệu chỉnh prior: trung bình xác suất = {p_adj.mean():.3f}")

Với label shift, nếu biết (hoặc ước lượng được) tỉ lệ lớp mới, có thể **hiệu chỉnh** xác suất bằng công thức Bayes như trên mà không cần train lại (chương 37 — calibration).

## 4. Concept drift — quy luật thay đổi

Kẻ gian thay đổi chiêu thức: trước đây giao dịch gian lận thường **số tiền lớn**; giờ chúng chuyển sang **nhiều giao dịch nhỏ**. Cùng một $X$ nhưng ý nghĩa của nó đã khác:

In [ ]:
def fraud_data(n, regime):
    amount = rng.lognormal(0, 1, n)
    n_tx = rng.poisson(2, n)
    logit = -4 + (1.5 * np.log(amount) if regime == "cũ" else 0.9 * n_tx - 1.2 * np.log(amount))
    y = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    return np.c_[np.log(amount), n_tx], y


X_old, y_old = fraud_data(20000, "cũ")
X_drift, y_drift = fraud_data(20000, "mới")
model = RandomForestClassifier(n_estimators=100, min_samples_leaf=20, random_state=0, n_jobs=-1).fit(X_old, y_old)
X_same, y_same = fraud_data(20000, "cũ")          # dữ liệu mới nhưng CÙNG quy luật
print(f"ROC-AUC trên dữ liệu cùng thời kỳ: {roc_auc_score(y_same, model.predict_proba(X_same)[:, 1]):.3f}")
print(f"ROC-AUC sau concept drift          : {roc_auc_score(y_drift, model.predict_proba(X_drift)[:, 1]):.3f}")

Concept drift không thể sửa bằng hiệu chỉnh đơn giản — cần **giám sát** liên tục và **train lại** với dữ liệu mới (chương 40 — MLOps).

## 5. Phát hiện shift

### 5.1 Kiểm định từng feature

So sánh phân phối từng feature giữa train và dữ liệu mới bằng **Kolmogorov–Smirnov test** (biến số) hoặc chi-square (biến phân loại); hoặc chỉ số **PSI** (Population Stability Index) phổ biến trong ngân hàng (chương 40).

In [ ]:
for name, sample_new in [("x khi triển khai (covariate shift)", x_deploy), ("x mới cùng phân phối", rng.uniform(0, 3, 200))]:
    res = stats.ks_2samp(x_train, sample_new)
    print(f"{name:38s} KS = {res.statistic:.3f}  p-value = {res.pvalue:.2e}")

### 5.2 Adversarial validation

Ý tưởng hay dùng trong thực tế (và Kaggle): train một classifier phân biệt **"đây là dữ liệu train hay dữ liệu mới?"**.
- AUC ≈ 0.5 → hai tập không phân biệt được → không có shift đáng kể.
- AUC cao → có shift; **feature importance** của classifier chỉ ra feature nào bị shift.

In [ ]:
from ml_course.datasets import make_house_prices

old = make_house_prices(n=2000, seed=0)
new = make_house_prices(n=2000, seed=1)
new["area_m2"] = new["area_m2"] * 0.8                # thị trường mới: nhà nhỏ hơn
new["age_years"] = new["age_years"] + 5               # và cũ hơn
cols = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]


def adversarial_auc(a, b):
    X = np.vstack([a[cols].to_numpy(), b[cols].to_numpy()])
    is_new = np.r_[np.zeros(len(a)), np.ones(len(b))]
    clf = RandomForestClassifier(n_estimators=200, min_samples_leaf=10, random_state=0, n_jobs=-1)
    proba = cross_val_predict(clf, X, is_new, cv=5, method="predict_proba")[:, 1]
    importances = clf.fit(X, is_new).feature_importances_
    return roc_auc_score(is_new, proba), dict(zip(cols, importances.round(3)))


print("Không shift:", adversarial_auc(old, make_house_prices(n=2000, seed=2)))
print("Có shift   :", adversarial_auc(old, new))

## 6. Xử lý

| Tình huống | Hướng xử lý |
|---|---|
| Covariate shift nhẹ | **importance weighting**: tăng trọng số các mẫu train giống dữ liệu mới (dùng xác suất từ adversarial classifier); thu thập dữ liệu ở vùng còn thiếu |
| Dữ liệu mới nằm ngoài vùng train | phát hiện "out-of-distribution", từ chối dự đoán hoặc chuyển cho con người |
| Label shift | hiệu chỉnh prior (mục 3), chọn lại threshold |
| Concept drift | giám sát hiệu năng + dữ liệu, **train lại định kỳ** hoặc khi phát hiện drift, ưu tiên dữ liệu gần đây |
| Mọi trường hợp | validation/test phải **giống dữ liệu khi triển khai** nhất có thể (vd: test = dữ liệu tháng gần nhất) |

## 🧠 Kiểm tra nhanh

1. Model nhận diện chữ viết tay train trên chữ người lớn, triển khai cho học sinh tiểu học. Đây là dạng shift nào?
2. ROC-AUC không đổi nhưng xác suất dự đoán trung bình lệch xa tỉ lệ thật — dạng shift nào phù hợp nhất?
3. Adversarial validation cho AUC = 0.93. Điều này nói lên gì và bước tiếp theo là gì?

## 📌 Tóm tắt

- Giả định ngầm của ML: train và triển khai cùng phân phối — thực tế thường bị vi phạm.
- Covariate shift ($P(X)$), label shift ($P(Y)$), concept drift ($P(Y \mid X)$).
- Phát hiện: KS/PSI từng feature, adversarial validation; giám sát liên tục khi triển khai.
- Xử lý: importance weighting, hiệu chỉnh prior, train lại, test set giống tương lai.

## 📚 Tài liệu tham khảo

- Quiñonero-Candela et al. — *Dataset Shift in Machine Learning* (MIT Press, 2009)
- Lipton, Wang, Smola — *Detecting and Correcting for Label Shift with Black Box Predictors* (ICML 2018)
- Chip Huyen — *Designing Machine Learning Systems* (2022), chương 8